# Flash Sales Incrementality: Evidence Synthesis

## The Question
Do Flash Sales generate incremental orders, and can we trust our measurement?

## The Problem
Flash Sales in Maverick cities (Dortmund, Dresden, Essen) **cannot be isolated** from 6+ concurrent campaigns (Smart Growth, SVP, Win Back, TFD, Prospect, Refer a Friend, Sales Enablement, Offline vouchering). Any city-level comparison captures the combined effect of all programmes.

## Three Lines of Evidence
1. **A/A Validation:** Does our BSTS method show zero uplift when no campaigns are active?
2. **Portfolio Measurement (BSTS):** What is the total Maverick programme effect? (Honest upper bound)
3. **Within-City DiD:** What is the marginal Flash Sales spike on top of always-running background campaigns?

No single approach gives a clean Flash Sales number. Together, they triangulate a defensible range.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)

import aa_config as cfg
import flash_sales_utils as fsu

client = bigquery.Client(project=cfg.PROJECT_ID)
print('Connected.')

## 2. A/A Validation: Is Our Method Biased?

We ran the full Flash Sales evaluation pipeline (MAVERICK city selection → correlation filter → hourly CausalImpact) on **historical non-campaign periods**. No Flash Sales were active. The expected result is zero uplift.

In [ ]:
df_aa = client.query(f"""
    SELECT treatment_city, window_start, window_end,
        n_controls, campaign_period_uplift, campaign_incr_orders
    FROM `{cfg.AA_RESULTS_TABLE}`
    WHERE technique = '{cfg.FLASH_SALES_TECHNIQUE}'
      AND campaign_period_uplift IS NOT NULL
    ORDER BY treatment_city, window_start
""").to_dataframe()

print(f'A/A runs: {len(df_aa)}')
print(f'Cities: {df_aa["treatment_city"].unique().tolist()}')
print(f'Windows: {df_aa["window_start"].nunique()}')

from scipy import stats

uplifts = df_aa['campaign_period_uplift'].values
n = len(uplifts)
mean_bias = np.mean(uplifts)
std_bias = np.std(uplifts, ddof=1)
ci_95 = stats.t.interval(0.95, df=n-1, loc=mean_bias,
                          scale=std_bias / np.sqrt(n))
ci_contains_zero = ci_95[0] <= 0 <= ci_95[1]

print(f'\nA/A Bias:')
print(f'  Mean:   {mean_bias:+.4f} ({mean_bias*100:+.1f}%)')
print(f'  Std:    {std_bias:.4f} ({std_bias*100:.1f}%)')
print(f'  95% CI: [{ci_95[0]:+.4f}, {ci_95[1]:+.4f}]')
print(f'  CI contains zero: {ci_contains_zero}')

if abs(mean_bias) > cfg.BIAS_THRESHOLD_HARD:
    aa_verdict = 'HARD FAIL'
elif abs(mean_bias) > cfg.BIAS_THRESHOLD_WARN or not ci_contains_zero:
    aa_verdict = 'WARNING'
else:
    aa_verdict = 'PASS'
print(f'  Verdict: {aa_verdict}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(uplifts, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Expected (zero)')
ax.axvline(mean_bias, color='orange', linestyle='-', linewidth=2,
           label=f'Observed mean: {mean_bias:+.3f}')
ax.axvspan(ci_95[0], ci_95[1], alpha=0.15, color='orange', label='95% CI')
ax.set_xlabel('Uplift (A/A period)')
ax.set_ylabel('Count')
ax.set_title('A/A Test: Uplift Distribution\n(should be centered on zero)')
ax.legend(fontsize=9)

# Per-city
ax = axes[1]
cities = sorted(df_aa['treatment_city'].unique())
data = [df_aa[df_aa['treatment_city']==c]['campaign_period_uplift'].values
        for c in cities]
bp = ax.boxplot(data, labels=cities, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.5)
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_ylabel('Uplift')
ax.set_title('A/A Uplift by Treatment City')

plt.tight_layout()
plt.show()

## 3. Power Assessment: Can We Detect Flash Sales Effects?

The A/A test tells us the **noise floor** of our method. The Minimum Detectable Effect (MDE) is the smallest real effect we can reliably distinguish from noise.

In [ ]:
mde = fsu.compute_mde(uplifts)

print(f'\nInterpretation:')
print(f'  If a Flash Sale generates >{mde*100:.0f}% uplift, '
      f'we can detect it with 80% probability.')
print(f'  If the true effect is <{mde*100:.0f}%, '
      f'our result may be indistinguishable from noise.')

# Visual: MDE vs plausible effect range
fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(['MDE\n(80% power)'], [mde*100], color='tomato', alpha=0.7, height=0.4)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Effect Size (%)')
ax.set_title('Minimum Detectable Effect vs. Plausible Flash Sales Range')
ax.set_xlim(-5, max(mde*100 + 10, 30))

# Add reference ranges (adjust these based on domain knowledge)
ax.axvspan(5, 15, alpha=0.1, color='green', label='Plausible Flash Sales range (5-15%)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

if mde < 0.15:
    print('Power assessment: SUFFICIENT for typical Flash Sales effects.')
else:
    print('Power assessment: INSUFFICIENT — results will be directional only.')

## 4. The Confounding Problem: Campaign Timeline

Flash Sales on Feb 27-28 overlap with 6+ concurrent Maverick campaigns in the same cities. Any city-level BSTS comparison captures the combined effect, not just Flash Sales.

In [ ]:
# Show campaign timeline for Essen (the primary Flash Sales city)
fig = fsu.plot_campaign_timeline(
    'Essen',
    date_range_start='2026-01-15',
    date_range_end='2026-04-01',
    highlight_dates=[('2026-02-27', '2026-02-28')])
plt.show()

# List what was active on Feb 27-28
print('\nCampaigns active in Essen on Feb 27-28 (Flash Sales dates):')
for c in fsu.get_concurrent_campaigns('Essen', '2026-02-27'):
    print(f"  - {c['campaign']} ({c['segment']}): {c['start']} to {c['end']}")

print('\nConclusion: The BSTS comparison of Essen vs control cities')
print('captures ALL of the above, not just Flash Sales.')

## 5. Evidence Line 2: Portfolio Measurement (BSTS)

Total Maverick programme incrementality across all 3 cities (Feb 4 - Mar 31).
This is the **upper bound** — it includes all campaigns, not just Flash Sales.

*Fill in after running `maverick_portfolio_evaluation.ipynb`*

In [ ]:
# Placeholder — fill with actual values from maverick_portfolio_evaluation.ipynb
portfolio_uplift = None       # e.g., 0.12
portfolio_incr_orders = None  # e.g., 5000

# Placeholder — fill with actual values from flash_sales_did_analysis.ipynb
did_pct_diff = None           # e.g., 0.08
did_diff_orders = None        # e.g., 200

if portfolio_uplift is not None:
    print(f'Portfolio (all Maverick campaigns):')
    print(f'  Uplift: {portfolio_uplift:+.1%}')
    print(f'  Incremental orders: {portfolio_incr_orders:+.0f}')
else:
    print('Portfolio results: PENDING')
    print('Run maverick_portfolio_evaluation.ipynb first.')

print()

if did_pct_diff is not None:
    print(f'DiD (marginal Flash Sales spike):')
    print(f'  Dinner-hour difference: {did_pct_diff:+.1%}')
    print(f'  Extra orders per Flash Sales day: {did_diff_orders:+.0f}')
else:
    print('DiD results: PENDING')
    print('Run flash_sales_did_analysis.ipynb first.')

print()
print('Interpretation:')
print('  Portfolio uplift = total Maverick effect (upper bound for Flash Sales)')
print('  DiD difference = marginal Flash Sales spike (lower bound)')
print('  True Flash Sales effect is somewhere in between.')

## 7. Limitations & What To Tell Stakeholders

**The honest answer:** We cannot provide a clean Flash Sales incrementality number. Here's why:

1. **Campaign isolation is impossible.** On any given Flash Sales day, 6+ other Maverick campaigns are simultaneously active in the same cities. No statistical method can separate the Flash Sales contribution from the portfolio.

2. **The original Flash Sales evaluation was invalid.** It had a 2-day post-period, a 2-month gap, Christmas in the baseline, aggregated controls, no hour filtering, and no contamination checks. Its reported incremental orders captured the entire Maverick programme, not Flash Sales.

3. **What we CAN say:**
   - The total Maverick portfolio generates X% uplift (from BSTS)
   - Flash Sales create a marginal dinner-hour spike of Y% on top of background campaigns (from DiD)
   - The BSTS method itself is unbiased under clean conditions (from A/A test)

4. **What we CANNOT say:**
   - "Flash Sales generated Z incremental orders" — because we can't separate them from Smart Growth, SVP, Win Back, etc.
   - Whether Flash Sales are cost-effective in isolation

5. **To get a clean Flash Sales number, you would need:**
   - A period where Flash Sales run but NO other Maverick campaigns are active, OR
   - An A/B test within the city (e.g., SERP interruptor randomisation), OR
   - Prospective randomisation across cities (Clustered Geo-Experiment approach)

In [ ]:
rows = []

# 1. A/A bias
rows.append({
    'Evidence Line': '1. A/A Validation',
    'Result': f'Mean bias = {mean_bias:+.3f}, CI = [{ci_95[0]:+.3f}, {ci_95[1]:+.3f}]',
    'Assessment': aa_verdict,
})

# 2. MDE
mde_ok = mde < 0.15
rows.append({
    'Evidence Line': '2. MDE (80% power)',
    'Result': f'{mde*100:.1f}%',
    'Assessment': 'Sufficient' if mde_ok else 'Insufficient — results are directional only',
})

# 3. Portfolio
if portfolio_uplift is not None:
    rows.append({
        'Evidence Line': '3. Portfolio BSTS (upper bound)',
        'Result': f'{portfolio_uplift:+.1%}, {portfolio_incr_orders:+.0f} orders',
        'Assessment': 'Positive' if portfolio_uplift > 0 else 'No effect detected',
    })
else:
    rows.append({
        'Evidence Line': '3. Portfolio BSTS (upper bound)',
        'Result': 'Not yet run',
        'Assessment': 'PENDING',
    })

# 4. DiD
if did_pct_diff is not None:
    rows.append({
        'Evidence Line': '4. DiD marginal spike (lower bound)',
        'Result': f'{did_pct_diff:+.1%}, {did_diff_orders:+.0f} orders/day',
        'Assessment': 'Flash Sales add value' if did_pct_diff > 0 else 'No marginal spike',
    })
else:
    rows.append({
        'Evidence Line': '4. DiD marginal spike (lower bound)',
        'Result': 'Not yet run',
        'Assessment': 'PENDING',
    })

# 5. Campaign isolation
rows.append({
    'Evidence Line': '5. Campaign isolation',
    'Result': f'{len(fsu.get_concurrent_campaigns("Essen", "2026-02-27"))} concurrent campaigns on Flash Sales dates',
    'Assessment': 'CANNOT isolate Flash Sales from portfolio',
})

# Overall
pending = any(r['Assessment'] == 'PENDING' for r in rows)
if pending:
    overall = 'PENDING — run remaining notebooks'
elif aa_verdict != 'PASS':
    overall = 'INCONCLUSIVE — method bias too high'
elif portfolio_uplift is not None and portfolio_uplift > 0 and did_pct_diff is not None and did_pct_diff > 0:
    overall = 'DIRECTIONAL — Flash Sales likely contribute, but magnitude cannot be isolated'
else:
    overall = 'INCONCLUSIVE'

rows.append({
    'Evidence Line': 'OVERALL VERDICT',
    'Result': '',
    'Assessment': overall,
})

df_cred = pd.DataFrame(rows)
print(df_cred.to_string(index=False))